# 03 — Baseline Model (Phase 1)
## Weighted SG Regression — The "Beat This or Stop" Benchmark

**Purpose:** Establish a simple, non-Bayesian baseline that the full model must outperform.

The baseline uses time-weighted average strokes gained as the player ability estimate. If the Bayesian model can't beat this, the added complexity isn't justified.

**What we measure:**
- RMSE of next-tournament SG predictions
- Rank correlation (does the model order players correctly?)
- Comparison against naive field average (predict 0 for everyone)


In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

from config.settings import Settings
from data.loader import DataLoader
from features.pipeline import FeaturePipeline
from models.baseline import BaselineModel, ModelRegistry

cfg = Settings()
loader = DataLoader(cfg)
rounds_df = loader.load_rounds()

rounds_df = rounds_df.rename(columns={
    "year":  "date",
    "dg_id": "player_id",
})
rounds_df["date"] = pd.to_datetime(rounds_df["date"].astype(str) + "-01-01")

# Split into train and holdout
if "season" in rounds_df.columns:
    train_df = rounds_df[rounds_df["season"].isin(cfg.TRAIN_SEASONS)]
    holdout_df = rounds_df[rounds_df["season"].isin(cfg.HOLDOUT_SEASONS)]
else:
    # Fallback: chronological split
    dates = pd.to_datetime(rounds_df["date"])
    cutoff = dates.quantile(0.75)
    train_df = rounds_df[dates <= cutoff]
    holdout_df = rounds_df[dates > cutoff]

print(f"Training:  {len(train_df):,} rounds ({train_df['player_id'].nunique():,} players)")
print(f"Holdout:   {len(holdout_df):,} rounds ({holdout_df['player_id'].nunique():,} players)")


In [ ]:
# --- Fit Baseline Model ---
pipeline = FeaturePipeline(cfg)
train_features = pipeline.run(train_df)

model = BaselineModel(cfg)
model.fit(train_features.player_features)
print("Baseline model fitted")


In [ ]:
# --- Evaluate on Holdout Events ---
# For each holdout event, predict SG for all players and compare to actual
holdout_events = holdout_df["event_id"].unique()
print(f"Evaluating on {len(holdout_events)} holdout events...")

predictions_all = []
actuals_all = []

for event_id in holdout_events:
    event_data = holdout_df[holdout_df["event_id"] == event_id]
    player_ids = event_data["player_id"].unique().tolist()

    preds_df = model.predict_tournament(player_ids)  # ← correct method name
    preds_dict = dict(zip(preds_df["player_id"], preds_df["ewma_sg_per_round"]))

    for pid in player_ids:
        if pid in preds_dict:
            actual_sg = event_data[event_data["player_id"] == pid]["sg_total"].mean()
            if not np.isnan(actual_sg):
                predictions_all.append(preds_dict[pid])
                actuals_all.append(actual_sg)

predictions_all = np.array(predictions_all)
actuals_all = np.array(actuals_all)

rmse = np.sqrt(np.mean((predictions_all - actuals_all) ** 2))
naive_rmse = np.sqrt(np.mean(actuals_all ** 2))  # Predict 0 for everyone
correlation = sp_stats.pearsonr(predictions_all, actuals_all)[0]

print(f"\n=== BASELINE RESULTS ===")
print(f"RMSE (baseline model): {rmse:.4f}")
print(f"RMSE (naive/zero):     {naive_rmse:.4f}")
print(f"Improvement over naive: {(1 - rmse/naive_rmse)*100:.1f}%")
print(f"Correlation (pred vs actual): {correlation:.4f}")


In [ ]:
# --- Prediction vs Actual Scatter Plot ---
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(predictions_all, actuals_all, alpha=0.15, s=10, color="#2563eb")
ax.plot([-3, 3], [-3, 3], "--", color="red", alpha=0.5, label="Perfect prediction")
ax.set_xlabel("Predicted SG (baseline)")
ax.set_ylabel("Actual SG")
ax.set_title(f"Baseline Model: Predicted vs Actual\nCorr={correlation:.3f}, RMSE={rmse:.3f}")
ax.set_xlim(-3, 3)
ax.set_ylim(-6, 6)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- Save Model ---
registry = ModelRegistry(cfg)
model_path = registry.save_model(
    model,
    name="baseline_v1",
    metadata={
        "type": "baseline",
        "train_seasons": cfg.TRAIN_SEASONS,
        "rmse": float(rmse),
        "correlation": float(correlation),
    },
)
print(f"Model saved to: {model_path}")


---
## ✅ Baseline Model Complete

**Benchmark established.** The Bayesian model (Notebook 04) must beat this RMSE and correlation.

**Interpretation:**
- If RMSE improvement over naive is < 5%, the data may not contain enough signal
- If correlation < 0.15, predicting individual tournament SG is very hard (expected!)
- Remember: we don't need to predict SG perfectly — we need to get *rankings* right enough to find mispriced odds

**Next step:** → **04_bayesian_model.ipynb**
